In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load
import os
import sys
import json
import random

import numpy as np 
import pandas as pd 

import librosa as lb
import librosa.feature as lf
import librosa.display as ld
import soundfile as sf
import kagglehub 


import matplotlib.pyplot as plt
from IPython.display import Audio
from tqdm import tqdm

import torch
import torchaudio
import torchaudio.transforms as T
from torch.utils.data import Dataset,DataLoader

# Kaggle Set-up
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
#os.environ['HF_TOKEN'] = user_secrets.get_secret("hf_access")
os.environ["KAGGLE_USERNAME"] = user_secrets.get_secret("kgg_user")
os.environ["KAGGLE_KEY"] = user_secrets.get_secret("kgg_key")

GENRES = ['blues', 'classical', 'country', 'disco', 'hiphop','jazz', 'metal', 'pop', 'reggae', 'rock'] 
SR = 22050
DURATION = 30

#============ Check for GPU availability ==============================#
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 Using device: {device}")

RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
if torch.cuda.is_available():
    print("GPU is available.Setting RANDOM_SEED .... ")
        # setting for both CPU and GPU
    torch.cuda.manual_seed_all(RANDOM_SEED)
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("   Running on CPU")
print("\n✅ Environment setup complete!")

import warnings
warnings.filterwarnings("ignore")

🚀 Using device: cuda
GPU is available.Setting RANDOM_SEED .... 
   GPU: Tesla T4
   Memory: 14.6 GB

✅ Environment setup complete!


In [2]:
class MusicDataset(Dataset):
    """
        This dataset takes file_paths as a list of all 15000
        paths of musics and load each music as waveform.
    """
    #,root_dir_path,file_names,sample_rate=SR
    def __init__(self,audios):
        self.audios = audios
    def __len__(self):
        return len(self.audios)
    def __getitem__(self,idx):
        waveform = self.audios[idx]
        return waveform
def extract_features_batch(waveforms, sr=22050):
    """
    waveforms: Tensor (batch_size, time) on CUDA
    returns: Tensor (batch_size, feature_dim)
    """
    waveforms = waveforms.squeeze(1)
    device = waveforms.device
    batch_size = waveforms.shape[0]
    # STFT (batched)
    stft = torch.stft(
        waveforms,
        n_fft=2048,
        hop_length=512,
        return_complex=True
    )  # (B, F, T)

    magnitude = stft.abs()
    power = magnitude ** 2

    # Frequency bins
    freqs = torch.linspace(0, sr/2, magnitude.shape[1], device=device)
    freqs = freqs.view(1, -1, 1)  # (1, F, 1)

    # Spectral Centroid
    centroid = (freqs * magnitude).sum(dim=1) / magnitude.sum(dim=1)
    centroid_mean = centroid.mean(dim=1)
    centroid_var = centroid.var(dim=1)

    # Spectral Bandwidth
    centroid_expanded = centroid.unsqueeze(1)
    bandwidth = torch.sqrt(
        ((freqs - centroid_expanded) ** 2 * magnitude).sum(dim=1)
        / magnitude.sum(dim=1)
    )
    bandwidth_mean = bandwidth.mean(dim=1)
    bandwidth_var = bandwidth.var(dim=1)

    #Spectral Rolloff (85%)
    cumulative = torch.cumsum(magnitude, dim=1)
    threshold = 0.85 * cumulative[:, -1:, :]
    rolloff = (cumulative >= threshold).float().argmax(dim=1)

    rolloff_mean = rolloff.float().mean(dim=1)
    rolloff_var = rolloff.float().var(dim=1)

    # RMS
    rms = torch.sqrt(torch.mean(waveforms ** 2, dim=1))

    # ZCR
    zcr = ((waveforms[:, 1:] * waveforms[:, :-1]) < 0).float().mean(dim=1)

    # MFCC (batched)
    mfcc_transform = T.MFCC(sample_rate=sr, n_mfcc=20).to(device)

    # MFCC expects (B, T)
    mfcc = mfcc_transform(waveforms)  # (B, 20, frames)

    mfcc_mean = mfcc.mean(dim=2)
    mfcc_var = mfcc.var(dim=2)

    # Concatenate All Features
    features = torch.cat([
        centroid_mean.unsqueeze(1),
        centroid_var.unsqueeze(1),
        bandwidth_mean.unsqueeze(1),
        bandwidth_var.unsqueeze(1),
        rolloff_mean.unsqueeze(1),
        rolloff_var.unsqueeze(1),
        rms.unsqueeze(1),
        zcr.unsqueeze(1),
        mfcc_mean,
        mfcc_var
    ], dim=1)

    return features         
print("✅")

✅


In [3]:
GENRES = ['blues', 'classical', 'country', 'disco', 'hiphop','jazz', 'metal', 'pop', 'reggae', 'rock'] 
g = "rock"
root_dir_path = f"/kaggle/input/datasets/akashkumbhakar/{g}-15000/"
music_file_names = [f"mashup_{i}.wav" for i in range(0,10000)]

audios = []
for i,f in tqdm(enumerate(music_file_names),total=len(music_file_names)):
    waveform, sr = torchaudio.load(root_dir_path + music_file_names[i])
    audios.append(waveform)

dataset = MusicDataset(audios)
data_loader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

100%|██████████| 10000/10000 [04:46<00:00, 34.90it/s]


In [4]:
all_features = []
for waveforms in tqdm(data_loader,desc="Extracting features ....",total=len(data_loader)):
    waveforms = waveforms.to(device)
    features = extract_features_batch(waveforms)
    all_features.append(features.cpu())
    
all_features = torch.cat(all_features, dim=0)
print(all_features.size())

Extracting features ....: 100%|██████████| 313/313 [00:31<00:00, 10.06it/s]

torch.Size([10000, 48])


In [5]:
feature_mat = all_features.numpy()
feature_mat.shape

(10000, 48)

In [6]:
def get_features_name():
    features_name = [
        "centroid_mean",
        "centroid_var",
        "bandwidth_mean",
        "bandwidth_var",
        "rolloff_mean",
        "rolloff_var",
        "rms",
        "zcr"
    ]
    for i in range(1,21):
        features_name.append(f"mfcc_mean{i}")
    for i in range(1,21):
        features_name.append(f"mfcc_var{i}")
        
    return features_name

df = pd.DataFrame(feature_mat,columns=get_features_name())
df.to_csv(f'{g}-10000.csv')
print("✅", df.shape)

✅ (10000, 48)


In [7]:
# Storing to kaggle hub
handle = f'akashkumbhakar/{g}-10000-csv'
local_dataset= f'/kaggle/working/{g}-10000.csv'

# Create a new dataset
kagglehub.dataset_upload(handle, local_dataset)

Uploading Dataset https://api.kaggle.com/datasets/akashkumbhakar/rock-10000-csv ...
Starting upload for file /kaggle/working/rock-10000.csv


Uploading: 100%|██████████| 4.87M/4.87M [00:00<00:00, 6.51MB/s]

Upload successful: /kaggle/working/rock-10000.csv (5MB)


Your dataset has been created.
Files are being processed...
See at: https://api.kaggle.com/datasets/akashkumbhakar/rock-10000-csv
